In [ ]:
import pandas as pd
import math
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [3]:
df_anime = pd.read_csv('anime.csv')
df_ratings = pd.read_parquet('ratings.parquet')

In [4]:
df_ratings

,user_id,anime_id,rating
0,0,0,8
1,0,1,6
2,0,2,9
3,0,3,10
4,0,4,9
...,...,...,...
6144921,47142,352,8
6144922,47142,51,7
6144923,47142,54,7
6144924,47142,1784,9


In [1]:
df_ratings

NameError: name 'df_ratings' is not defined

In [24]:
print(df_ratings['rating'].min())
print(df_ratings['user_id'].min())
print(df_ratings['anime_id'].min())

1
0
0


In [ ]:
NUM_USERS = df_ratings['user_id'].nunique(dropna=True)
NUM_ITEMS = df_ratings['anime_id'].nunique(dropna=True)
print(f'Hay {NUM_USERS} usuarios y {NUM_ITEMS} animes')

X = df_ratings[['user_id', 'anime_id']]
y = df_ratings['rating']

# Hacemos el split para el modelo de recomendación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Filas de entrenamiento: {X_train.shape[0]:,}")
print(f"Filas de test: {X_test.shape[0]:,}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entrenando en: {device}") # Debería salir 'cuda'

In [ ]:

latent_dim = 5
epochs = 10

In [ ]:
class GMFModel(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        self.fc = nn.Linear(latent_dim, 1)

    def forward(self, user_ids, item_ids):
        u_emb = self.user_embedding(user_ids)
        i_emb = self.item_embedding(item_ids)
        interact = torch.mul(u_emb, i_emb)
        return self.fc(interact).flatten()
# Instanciamos el modelo
GMF = GMFModel(NUM_USERS, NUM_ITEMS, latent_dim)
print(GMF)

GMFModel(
  (user_embedding): Embedding(47144, 5)
  (item_embedding): Embedding(6533, 5)
  (fc): Linear(in_features=5, out_features=1, bias=True)
)


###  Simple Perceptron

In [ ]:
# Configuración de optimizador y pérdida
optimizer = torch.optim.Adam(GMF.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Mover el modelo a la GPU
GMF = GMF.to(device)

# Preparación de datos (Corregido el acceso a columnas y conversión a .values)
users_t = torch.tensor(X_train['user_id'].values, dtype=torch.long)
items_t = torch.tensor(X_train['anime_id'].values, dtype=torch.long)
ratings_t = torch.tensor(y_train.values, dtype=torch.float32)

dataset = TensorDataset(users_t, items_t, ratings_t)
# Subimos el batch_size a 2048 para aprovechar la VRAM de la 3060 Ti
loader = DataLoader(dataset, batch_size=2048, shuffle=True)

# Bucle de entrenamiento
for epoch in range(epochs):
    GMF.train()
    total_loss = 0
    for batch_u, batch_i, batch_r in loader:
        # Enviar cada batch a la GPU
        batch_u = batch_u.to(device)
        batch_i = batch_i.to(device)
        batch_r = batch_r.to(device)

        optimizer.zero_grad()

        # Predicción
        outputs = GMF(batch_u, batch_i)
        # Cálculo de error
        loss = loss_fn(outputs, batch_r)
        # Backpropagation
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Época {epoch+1}/{epochs} - Pérdida (MSE): {total_loss/len(loader):.4f}")

NameError: name 'X_train' is not defined

In [ ]:
GMF.eval()
with torch.no_grad():
    # Convertimos los datos de test a tensores y los enviamos a la GPU
    users_test = torch.tensor(X_test['user_id'].values, dtype=torch.long).to(device)
    items_test = torch.tensor(X_test['anime_id'].values, dtype=torch.long).to(device)

    # Devolvemos las predicciones a la CPU para poder usar numpy/sklearn
    y_pred = GMF(users_test, items_test).cpu().numpy()
y_pred

In [ ]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_pred)

### Multilayer Perceptron

In [ ]:
latent_dim = 16
epochs = 5

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, user_ids, item_ids):
        u_emb = self.user_embedding(user_ids)
        i_emb = self.item_embedding(item_ids)
        vector = torch.cat([u_emb, i_emb], dim=-1)
        return self.mlp(vector).flatten()

MLP = MLPModel(NUM_USERS, NUM_ITEMS, latent_dim)
print(MLP)

In [ ]:
optimizer = torch.optim.Adam(MLP.parameters())
loss_fn = nn.MSELoss()

users_t = torch.tensor(X_train[0], dtype=torch.long)
items_t = torch.tensor(X_train[1], dtype=torch.long)
ratings_t = torch.tensor(y_train, dtype=torch.float32)
dataset = TensorDataset(users_t, items_t, ratings_t)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

# Entrenamos el MLP
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for b_u, b_i, b_r in loader:
        optimizer.zero_grad()
        preds = model(b_u, b_i)
        loss = loss_fn(preds, b_r)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Época {epoch+1}: Pérdida {total_loss/len(loader):.4f}")

In [ ]:
MLP.eval()
with torch.no_grad():
    users_test = torch.tensor(X_test[0], dtype=torch.long)
    items_test = torch.tensor(X_test[1], dtype=torch.long)
    y_pred = MLP(users_test, items_test).numpy()
y_pred

In [ ]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_pred)